# NYC Traffic Accidents - Exploratory Data Analysis (Python)

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"

clean_path = "../data/nyc_traffic_accidents_clean.csv"
df = pd.read_csv(clean_path)
df.shape, df.dtypes.head()

((74881, 28),
 crash_date     object
 borough        object
 zip_code      float64
 latitude      float64
 longitude     float64
 dtype: object)

## 1. Monthly Trends

In [2]:
monthly = (
    df.groupby(["month", "month_name"])
      .agg(
          total_crashes=("collision_id", "count"),
          fatal_crashes=("is_fatal", "sum")
      )
      .reset_index()
      .sort_values("month")
)
total_fatal_crashes = sum(monthly["fatal_crashes"])
monthly["fatal_rate_pct"] = round((monthly["fatal_crashes"] / total_fatal_crashes) * 100, 2)
monthly

,month,month_name,total_crashes,fatal_crashes,fatal_rate_pct
0,1,January,14287,17,12.59
1,2,February,13684,20,14.81
2,3,March,11057,8,5.93
3,4,April,4116,13,9.63
4,5,May,6149,12,8.89
5,6,June,7616,30,22.22
6,7,July,9225,20,14.81
7,8,August,8747,15,11.11


In [3]:
fig = go.Figure()

# Bars: total crashes
fig.add_trace(go.Bar(
    x=monthly["month_name"],
    y=monthly["total_crashes"],
    name="Total Crashes"
))

# Line: fatality rate
fig.add_trace(go.Scatter(
    x=monthly["month_name"],
    y=monthly["fatal_rate_pct"],
    name="Share of Fatal Crashes (%)",
    mode="lines+markers",
    yaxis="y2"
))

fig.update_layout(
    title="Monthly Collision Trends (Jan-Aug 2020)",
    xaxis_title="Month",
    yaxis_title="Number of Crashes",
    yaxis2=dict(
        title="Share (%)",
        overlaying="y",
        side="right"
    ),
    barmode="group",
    template="plotly_white"
)

fig.show()

## 2. Day-of-Week Analysis

In [4]:
weekday_order = [
    "Sunday", "Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday"
]

df["day_of_week"] = pd.Categorical(
    df["day_of_week"],
    categories=weekday_order,
    ordered=True
)

dow = (
    df.groupby("day_of_week", observed=True)
      .agg(
          total_crashes=("collision_id", "count"),
          fatal_crashes=("is_fatal", "sum")
      )
      .reset_index()
      .sort_values("day_of_week")
)

dow["fatal_rate_pct"] = round((dow["fatal_crashes"] / total_fatal_crashes) * 100, 2)
dow


,day_of_week,total_crashes,fatal_crashes,fatal_rate_pct
0,Sunday,9003,22,16.30
1,Monday,10511,16,11.85
2,Tuesday,10613,25,18.52
3,Wednesday,10638,15,11.11
4,Thursday,11244,13,9.63
5,Friday,12271,16,11.85
6,Saturday,10601,28,20.74


In [5]:
fig = go.Figure()

# Bars: total crashes
fig.add_trace(go.Bar(
    x=dow["day_of_week"],
    y=dow["total_crashes"],
    name="Total Crashes"
))

# Line: fatality rate
fig.add_trace(go.Scatter(
    x=dow["day_of_week"],
    y=dow["fatal_rate_pct"],
    name="Share of Fatal Crashes (%)",
    mode="lines+markers",
    yaxis="y2"
))

fig.update_layout(
    title="Crashes by Day of Week",
    xaxis_title="Day of Week",
    yaxis_title="Number of Crashes",
    yaxis2=dict(
        title="Share (%)",
        overlaying="y",
        side="right"
    ),
    template="plotly_white"
)

fig.show()

## 3. Hour-of-Day Patterns

In [6]:
hourly = (
    df.groupby("hour")
      .agg(
          total_crashes=("collision_id", "count"),
          fatal_crashes=("is_fatal", "sum")
      )
      .reset_index()
      .sort_values("hour")
)

hourly.head()

,hour,total_crashes,fatal_crashes
0,0,2948,7
1,1,1474,7
2,2,1139,2
3,3,989,7
4,4,975,7


In [7]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=hourly["hour"],
    y=hourly["total_crashes"],
    name="Total Crashes",
    mode="lines+markers"
))

fig.update_layout(
    title="Crashes by Hour of Day",
    xaxis=dict(
        title="Hour of Day",
        dtick=1,
        tickvals=hourly["hour"]
    ),
    yaxis_title="Total Crashes",
    template="plotly_white"
)

fig.show()

In [8]:
weekday_df = df[~df["is_weekend"]]
weekend_df = df[df["is_weekend"]]

weekday_hourly = weekday_df.groupby("hour")["collision_id"].count() / weekday_df["crash_date"].nunique()
weekend_hourly = weekend_df.groupby("hour")["collision_id"].count() / weekend_df["crash_date"].nunique()

hourly_avg = pd.DataFrame({
    "hour": range(24),
    "weekday_avg": weekday_hourly.reindex(range(24), fill_value=0).values,
    "weekend_avg": weekend_hourly.reindex(range(24), fill_value=0).values
})

hourly_avg.head()

,hour,weekday_avg,weekend_avg
0,0,11.052023,15.014493
1,1,4.809249,9.304348
2,2,3.520231,7.681159
3,3,2.907514,7.043478
4,4,3.069364,6.434783


In [9]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=hourly_avg["hour"],
    y=hourly_avg["weekday_avg"],
    name="Weekday Avg",
    mode="lines+markers"
))

fig.add_trace(go.Scatter(
    x=hourly_avg["hour"],
    y=hourly_avg["weekend_avg"],
    name="Weekend Avg",
    mode="lines+markers"
))

fig.update_layout(
    title="Average Crashes by Hour",
    xaxis=dict(
        title="Hour of Day",
        dtick=1,
        tickvals=hourly_avg["hour"]
    ),
    yaxis_title="Avg Crashes per Day",
    template="plotly_white"
)

fig.show()

In [10]:
radar_df = hourly_avg.copy()
radar_df.loc[24] = radar_df.loc[0] # close the loop
radar_df["hour_label"] = radar_df["hour"].astype(int).astype(str)

fig = go.Figure()

fig.add_trace(go.Scatterpolar(
    r=radar_df["weekday_avg"],
    theta=radar_df["hour_label"],
    name="Weekday Avg",
))

fig.add_trace(go.Scatterpolar(
    r=radar_df["weekend_avg"],
    theta=radar_df["hour_label"],
    name="Weekend Avg",
))

fig.update_layout(
    title="Radar Chart - Weekday vs Weekend Hourly Patterns",
    template="plotly_white",
    polar=dict(
        radialaxis=dict(visible=True),
        angularaxis=dict(direction="clockwise")
    )
)

fig.show()

## 4. Day × Hour Heatmap

In [11]:
day_hour = (
    df.groupby(["day_of_week", "hour"], observed=True)["collision_id"]
      .count()
      .reset_index()
      .pivot(index="day_of_week", columns="hour", values="collision_id")
      .reindex(index=weekday_order)  # just to be explicit
)

all_hours = list(range(24))
day_hour = day_hour.reindex(columns=all_hours, fill_value=0)

day_hour

hour,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
day_of_week,,,,,,,,,,,,,,,,,,,,,
Sunday,510,312,271,254,232,195,182,165,231,241,...,615,537,566,539,526,462,407,356,365,292
Monday,385,200,126,112,117,142,310,407,591,552,...,682,690,746,736,658,494,373,325,278,219
Tuesday,333,134,88,96,78,159,294,433,643,576,...,719,654,799,750,693,509,444,326,339,264
Wednesday,367,154,129,106,103,176,258,407,563,509,...,741,730,756,743,688,520,402,378,366,259
Thursday,391,144,116,78,118,148,316,445,679,593,...,722,683,814,778,703,556,454,399,340,283
Friday,436,200,150,111,115,178,315,383,648,638,...,815,781,870,804,800,628,574,472,430,428
Saturday,526,330,259,232,212,180,193,223,323,330,...,722,602,668,624,628,569,484,455,439,419


In [12]:
fig = px.imshow(
    day_hour,
    labels=dict(
        x="Hour of Day",
        y="Day of Week",
        color="Number of Crashes"
    ),
    x=day_hour.columns,
    y=day_hour.index,
    aspect="auto",
    color_continuous_scale="Reds"
)

fig.update_layout(
    title="Crashes Heatmap - Day of Week × Hour of Day",
    xaxis=dict(
        tickvals=day_hour.columns
    ),
    template="plotly_white"
)

fig.show()

## 5. High-Risk Locations

In [13]:
# streets = df[df["on_street_name"].notna()].copy()
# streets["on_street_name"] = streets["on_street_name"].str.strip()

# top_streets = (
#     streets.groupby("on_street_name")
#         .agg(total_crashes=("collision_id", "count"))
#         .reset_index()
#         .sort_values("total_crashes", ascending=False)
#         .head(10)
# )

# # Add % of total collisions
# total_crashes_citywide = len(df)
# top_streets["pct_of_total"] = (top_streets["total_crashes"] / total_crashes_citywide) * 100

# top_streets

## 6. Contributing Factors
## 7. Vehicle Types & Victims